In [ ]:
import schemdraw
import schemdraw.elements as elm

# Mapeia orientação textual para direção lógica do Schemdraw
direcoes = {
    'right': lambda comp: comp.right(),
    'left': lambda comp: comp.left(),
    'up': lambda comp: comp.up(),
    'down': lambda comp: comp.down(),
}

def desenhar_circuito(componentes):
    with schemdraw.Drawing() as d:
        for comp in componentes:
            tipo = comp.get("tipo", "")
            valor = comp.get("valor", "")
            orientacao = comp.get("orientacao", "right").lower()

            # Define a direção (padrão: right)
            direcao = direcoes.get(orientacao, direcoes['right'])

            # Adiciona o componente com a direção correta
            if tipo == "resistor":
                d += direcao(elm.Resistor().label(valor))
            elif tipo == "capacitor":
                d += direcao(elm.Capacitor().label(valor))
            elif tipo == "fonte":
                d += direcao(elm.SourceV().label(valor))
            elif tipo == "indutor":
                d += direcao(elm.Inductor().label(valor))
            elif tipo == "terra":
                d += elm.Ground()
            elif tipo == "fio":
                d += direcao(elm.Line())
            else:
                raise ValueError(f"Tipo desconhecido: {tipo}")

        d.draw()

componentes = [
    {"tipo": "fonte", "valor": "12V", "orientacao": "up"},
    {"tipo": "resistor", "valor": "100kΩ", "orientacao": "right"},
    {"tipo": "capacitor", "valor": "0.5μF", "orientacao": "down"},
    {"tipo": "fio", "orientacao": "left"},
    {"tipo": "terra"}
]

desenhar_circuito(componentes)



In [ ]:
import schemdraw
import schemdraw.elements as elm

# Dicionário de orientação Jedi
direcoes = {
    'right': lambda comp: comp.right(),
    'left': lambda comp: comp.left(),
    'up': lambda comp: comp.up(),
    'down': lambda comp: comp.down()
}

def desenhar_jedi(componentes):
    with schemdraw.Drawing() as d:
        for comp in componentes:
            tipo = comp.get("tipo")
            valor = comp.get("valor", "")
            orientacao = comp.get("orientacao", "right").lower()
            xy = comp.get("xy", None)  # posição absoluta
            anchor = comp.get("anchor", None)

            # Escolhe direção ou posição absoluta
            func_orient = direcoes.get(orientacao, direcoes['right'])

            # Base do componente com rótulo
            if tipo == "resistor":
                base = elm.Resistor().label(valor)
            elif tipo == "capacitor":
                base = elm.Capacitor().label(valor)
            elif tipo == "fonte":
                base = elm.SourceV().label(valor)
            elif tipo == "indutor":
                base = elm.Inductor().label(valor)
            elif tipo == "terra":
                base = elm.Ground()
            elif tipo == "fio":
                base = elm.Line()
            else:
                raise ValueError(f"Componente desconhecido: {tipo}")

            # Se coordenadas são fornecidas, usa elas
            if xy:
                d += base.at(xy).anchor(anchor if anchor else 'start')
            else:
                d += func_orient(base)

        d.draw()

componentes = [
    {"tipo": "fonte", "valor": "5V", "xy": (0, 0), "anchor": "start"},

    {"tipo": "resistor", "valor": "1kΩ", "orientacao": "right"},
    {"tipo": "resistor", "valor": "5kΩ", "orientacao": "right"},

    {"tipo": "capacitor", "valor": "10μF", "orientacao": "down"},

    {"tipo": "fio", "orientacao": "left"},
    {"tipo": "fio", "orientacao": "left"},

    {"tipo": "terra", "xy": (0, 0), "anchor": "start"}
]

desenhar_jedi(componentes)


In [ ]:
import schemdraw
import schemdraw.elements as elm
import sympy as sp


class CircuitoCC:
    def __init__(self):
        self.componentes = []
        self.equacoes = []
        self.vars = []
        self.symbols = {}
        self.nos = set()
        self.historico = []

    def add_resistor(self, nome, node_a, node_b, valor):
        R = sp.Symbol(nome) if isinstance(valor, str) else valor
        self.componentes.append({'tipo': 'R', 'nome': nome, 'nós': (node_a, node_b), 'valor': R})
        self.symbols[nome] = R
        self.nos.update([node_a, node_b])

    def add_fonte(self, nome, node_a, node_b, valor):
        V = sp.Symbol(nome) if isinstance(valor, str) else valor
        self.componentes.append({'tipo': 'V', 'nome': nome, 'nós': (node_a, node_b), 'valor': V})
        self.symbols[nome] = V
        self.nos.update([node_a, node_b])

    def aplicar_lei_ohm(self):
        print("\n⚡ LEI DE OHM:")
        for comp in self.componentes:
            if comp['tipo'] == 'R':
                i = sp.Symbol(f'I_{comp["nome"]}')
                v = comp['valor'] * i
                eq = sp.Eq(v, comp['valor'] * i)
                print(f"➤ {eq}")
                self.equacoes.append(eq)
                self.vars.append(i)
        self._registrar_operacao("Lei de Ohm")

    def aplicar_lkt(self):
        print("\n🧠 LEI DAS TENSÕES (LKT):")
        eq = 0
        for comp in self.componentes:
            i = sp.Symbol(f'I_{comp["nome"]}')
            if comp['tipo'] == 'V':
                eq += comp['valor']
            elif comp['tipo'] == 'R':
                eq -= comp['valor'] * i
        lkt_eq = sp.Eq(eq, 0)
        print(f"➤ {lkt_eq}")
        self.equacoes.append(lkt_eq)
        self._registrar_operacao("LKT - Lei das Tensões")

    def aplicar_lkc(self):
        print("\n🔁 LEI DAS CORRENTES (LKC):")
        for no in self.nos:
            if no == '0':
                continue
            eq = 0
            for comp in self.componentes:
                if no in comp['nós']:
                    i = sp.Symbol(f'I_{comp["nome"]}')
                    if comp['nós'][0] == no:
                        eq -= i  # saindo
                    elif comp['nós'][1] == no:
                        eq += i  # entrando
            if eq != 0:
                lkc_eq = sp.Eq(eq, 0)
                print(f"➤ {lkc_eq}")
                self.equacoes.append(lkc_eq)
        self._registrar_operacao("LKC - Lei das Correntes")

    def resolver_matriz(self):
        print("\n🧮 RESOLUÇÃO DO SISTEMA:")
        sol = sp.solve(self.equacoes, self.vars, dict=True)
        print(sol)
        for s in sol:
            for var, val in s.items():
                print(f"{var} = {sp.N(val)} A")
        return sol

    def resistencia_equivalente(self):
        # Suporte básico para malha simples série
        total = sum([comp['valor'] for comp in self.componentes if comp['tipo'] == 'R'])
        print(f"\n🔌 Resistência equivalente (série): Req = {total} Ω")
        return total

    def desenhar(self):
        with schemdraw.Drawing() as d:
            for comp in self.componentes:
                if comp['tipo'] == 'V':
                    d += elm.SourceV().label(f"{comp['nome']} = {comp['valor']}")
                elif comp['tipo'] == 'R':
                    d += elm.Resistor().label(f"{comp['nome']} = {comp['valor']}")
            d += elm.Line().up()

    def _registrar_operacao(self, titulo):
        estado = {
            'componentes': list(self.componentes),
            'equacoes': list(self.equacoes),
            'titulo': titulo
        }
        self.historico.append(estado)


# -------------------------------
# EXEMPLO COM VALORES REAIS
# -------------------------------

c = CircuitoCC()

# 🔌 Fonte de 12V, dois resistores: 100Ω e 200Ω
c.add_fonte('V', 'n1', '0', 12)
c.add_resistor('R1', 'n1', 'n2', 100)
c.add_resistor('R2', 'n2', '0', 200)

# Aplicando as leis
c.aplicar_lei_ohm()
c.aplicar_lkt()
c.aplicar_lkc()

# Resolvendo
c.resolver_matriz()

# Req total
c.resistencia_equivalente()

# Desenhando o circuito
c.desenhar()


In [ ]:
import schemdraw
import schemdraw.elements as elm

with schemdraw.Drawing():
    elm.Resistor()
    elm.Capacitor()
    elm.Diode()
    elm.Inductor()


    
    elm.Transistor()
    elm.OpAmp()
    elm.LED()
    elm.Relay()
    elm.Motor()
    elm.Battery()
    elm.Switch()
    elm.Fuse()
    elm.Crystal()
    elm.Oscillator()

In [ ]:
import schemdraw
import schemdraw.elements as elm


class DesenhadorCircuito:
    def __init__(self, r1_valor, c1_valor, v1_valor):
        self.r1 = r1_valor
        self.c1 = c1_valor
        self.v1 = v1_valor

    def desenhar_rc_malha(self):
        with schemdraw.Drawing() as d:
            d += elm.Resistor().label(f'{self.r1}')
            d += elm.Capacitor().down().label(f'{self.c1}μF', loc='bottom')
            d += elm.Line().left()
            d += elm.Ground()
            d += elm.SourceV().up().label(f'{self.v1}')


with schemdraw.Drawing():
    elm.Resistor().label('100KΩ')
    elm.Capacitor().down().label('0.5μF', loc='bottom')
    elm.Line().left()
    elm.Ground()
    elm.SourceV().up().label('12V')

In [ ]:



# Variáveis de entrada (como num exercício de livro)
r1 = "100kΩ"
c1 = 0.5   # em microfarads
v1 = "12V"

# Criar e desenhar circuito
desenho = DesenhadorCircuito(r1, c1, v1)
desenho.desenhar_rc_malha()

